# Disaster Assessment — Orchestration Notebook

Edit **only the PATH CONFIG cell** below when switching between Colab and local VS Code.
All model logic lives in the `.py` modules.

## 0 — Installation (run once)

In [ ]:
%pip install -U "rasterio>=1.4.3" "shapely>=2.0.3" albumentations opencv-python-headless scikit-learn tqdm
# DenseCRF (optional — skip if not needed)
# !pip install --quiet --upgrade --force-reinstall --no-deps git+https://github.com/lucasb-eyer/pydensecrf.git

## 1 — Colab / local import block

Choose **one** of Option A or Option B, comment out the other.

In [ ]:
import sys

# ── Option A: running on Colab ──────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# !git clone https://github.com/YOUR_USERNAME/disaster-assessment.git
# sys.path.insert(0, 'disaster-assessment')

# ── Option B: running locally in VS Code ─────────────────────────────────────
# (notebook lives inside the disaster-assessment/ folder, so '.' is enough)
if '.' not in sys.path:
    sys.path.insert(0, '.')

## 2 — PATH CONFIG  ← **only cell to edit between environments**

In [ ]:
from pathlib import Path

# ── Colab paths ──────────────────────────────────────────────────────────────
# XBD_ARCHIVE     = Path('/content/drive/MyDrive/TrainResNet_multiclass_XBD/train_images_labels_targets.tar')
# XBD_ROOT        = Path('/content/datasets/xbd2')
# UKR_ROOT        = Path('/content/drive/MyDrive/damage_assessment_ukraine/segmentation')
# CLS_ROOT        = Path('/content/drive/MyDrive/damage_assessment_ukraine/classification')
# XBD_CROPS_ROOT  = Path('/content/datasets/xbd_crops/train')
# CLS_RUNS_ROOT   = Path('/content/drive/MyDrive/xbd_final_runs')
# WAR_PRE_DIR     = Path('/content/drive/MyDrive/WarImagery/pre')
# WAR_POST_DIR    = Path('/content/drive/MyDrive/WarImagery/post')
# SEG_CKPT        = Path('/content/drive/MyDrive/damage_assessment_ukraine/segmentation/outputs/20250910-175901/best_model_kolega.pth')
# CLS_CKPT        = Path('/content/drive/MyDrive/xbd_final_runs/ablation_siam_resnet50_sz256_seed42/artifacts/finetune_kolega_f1on0/best_ft.pth')

# ── Local paths ───────────────────────────────────────────────────────────────
XBD_ARCHIVE     = Path('/Volumes/MyDrive/datasets/train_images_labels_targets.tar')
XBD_ROOT        = Path('/Volumes/MyDrive/datasets/xbd2')
UKR_ROOT        = Path('/Volumes/MyDrive/datasets/damage_assessment_ukraine/segmentation')
CLS_ROOT        = Path('/Volumes/MyDrive/datasets/damage_assessment_ukraine/classification')
XBD_CROPS_ROOT  = Path('/Volumes/MyDrive/datasets/xbd_crops/train')
CLS_RUNS_ROOT   = Path('/Volumes/MyDrive/runs/xbd_final_runs')
WAR_PRE_DIR     = Path('/Volumes/MyDrive/datasets/WarImagery/pre')
WAR_POST_DIR    = Path('/Volumes/MyDrive/datasets/WarImagery/post')
SEG_CKPT        = Path('/Volumes/MyDrive/runs/seg/best_model_kolega.pth')
CLS_CKPT        = Path('/Volumes/MyDrive/runs/cls/best_ft.pth')

## 3 — Shared setup

In [ ]:
import time
from utils import set_seed, get_device

set_seed(1337)
device = get_device()
print('Device:', device)

---
## PART A — XBD Segmentation

### A1 — Extract dataset

In [ ]:
XBD_ROOT.mkdir(parents=True, exist_ok=True)
!tar --exclude='._*' --exclude='__MACOSX' -xaf "{XBD_ARCHIVE}" -C "{XBD_ROOT}"

### A2 — Rasterize labels → binary masks

In [ ]:
from preprocess import rasterize_xbd_labels, normalise_masks

XBD_IMAGES  = XBD_ROOT / 'train' / 'images'
XBD_LABELS  = XBD_ROOT / 'train' / 'labels'
XBD_TARGETS = XBD_ROOT / 'targets_border0'
XBD_MASKS   = XBD_ROOT / 'masks_binary_b2'

rasterize_xbd_labels(str(XBD_LABELS), str(XBD_IMAGES), str(XBD_TARGETS))
normalise_masks(str(XBD_TARGETS), str(XBD_IMAGES), str(XBD_MASKS))

### A3 — Build manifest

In [ ]:
from dataset import build_xbd_manifest

XBD_MANIFEST = XBD_ROOT / 'manifest.csv'
build_xbd_manifest(str(XBD_IMAGES), str(XBD_MASKS), str(XBD_MANIFEST), val_frac=0.20)

### A4 — Train segmentation model (XBD)

In [ ]:
from dataset import SplitConfig, build_loaders_xbd
from train import train_single_manifest

RUN_DIR_XBD = XBD_ROOT / 'outputs' / time.strftime('%Y%m%d-%H%M%S')

model, train_loader, val_loader, best_iou, best_epoch = train_single_manifest(
    manifest_csv    = str(XBD_MANIFEST),
    build_loaders_fn= build_loaders_xbd,
    device          = device,
    run_dir         = RUN_DIR_XBD,
    split_cfg       = SplitConfig(split_col='split', train_tag='train', val_tag='val'),
    backbone        = 'resnet34',
    pretrained      = True,
    epochs          = 20,
    batch_size      = 8,
    lr              = 1e-4,
    num_workers     = 2,
    patience        = 8,
)
print(f'Best IoU: {best_iou:.4f} at epoch {best_epoch}')

### A5 — Evaluate XBD segmentation

In [ ]:
import torch
from models import ResUNet
from eval import run_test_block_optimized

XBD_SEG_CKPT = RUN_DIR_XBD / 'best_model.pth'
state = torch.load(XBD_SEG_CKPT, map_location=device)
model.load_state_dict(state); model.eval()

metrics, best_t = run_test_block_optimized(
    model=model, device=device, val_loader=val_loader, sweep_on_val=True, n_vis=6,
    save_dir=str(RUN_DIR_XBD / 'eval_viz')
)
print('Best t:', best_t, '| Metrics:', metrics)

---
## PART B — Ukraine Segmentation (fine-tune from XBD)

### B1 — Build Kolega segmentation manifest

In [ ]:
from dataset import build_kolega_seg_manifest

CITIES       = ['kamianka_data', 'popasna_data', 'yakovlivka_data']
COMMON_SPLIT = {
    'train': ['fold_2','fold_3','fold_5','fold_6','fold_7','fold_8','fold_9'],
    'val':   ['fold_1','fold_4'],
    'test':  ['fold_0'],
}
UKR_MANIFEST = UKR_ROOT / 'manifest_kolega.csv'
build_kolega_seg_manifest(str(UKR_ROOT), CITIES, COMMON_SPLIT, str(UKR_MANIFEST))

### B2 — Train Ukraine segmentation model

In [ ]:
from dataset import build_loaders_ukr

RUN_DIR_UKR = UKR_ROOT / 'outputs' / time.strftime('%Y%m%d-%H%M%S')

model_ukr, tl, vl, best_iou_ukr, best_ep_ukr = train_single_manifest(
    manifest_csv     = str(UKR_MANIFEST),
    build_loaders_fn = build_loaders_ukr,
    device           = device,
    run_dir          = RUN_DIR_UKR,
    split_cfg        = SplitConfig(split_col='split', train_tag='train', val_tag='val'),
    backbone         = 'resnet34',
    pretrained       = True,
    epochs           = 40,
    batch_size       = 4,
    lr               = 1e-4,
    num_workers      = 2,
    init_ckpt        = str(XBD_SEG_CKPT),   # fine-tune from XBD checkpoint
    freeze_encoder_epochs = 2,
    patience         = 8,
    sweep_tmin       = 0.75, sweep_tmax=0.95, sweep_steps=41,
    eval_postproc    = True, min_component=64,
)
print(f'Best Ukraine IoU: {best_iou_ukr:.4f} at epoch {best_ep_ukr}')

---
## PART C — XBD Damage Classification

### C1 — Build XBD crops manifest  (run preprocess.py crop pipeline first)

In [ ]:
# The crop manifest is generated by preprocess.py (crop_xbd_to_manifest).
# Point XBD_CROPS_ROOT to your crop output folder.
XBD_CLS_MANIFEST = XBD_CROPS_ROOT / 'manifest.csv'
import pandas as pd
df_check = pd.read_csv(XBD_CLS_MANIFEST)
print('Crop manifest shape:', df_check.shape)
print(df_check.head(3))

### C2 — Train Siamese classification model (XBD)

In [ ]:
import os
from train import train_one_run

BASE_CFG = {
    'SEED': 1337,
    'MANIFEST': str(XBD_CLS_MANIFEST),
    'INPUT_SIZE': 256,
    'NUM_CLASSES': 4,
    'BATCH_SIZE': 128,
    'TARGET_EFF_BS': 128,
    'FREEZE_EPOCHS': 2,
    'EPOCHS': 8,
    'PATIENCE': 4,
    'BACKBONE': 'resnet50',
    'DROPOUT': 0.40,
    'LR_HEAD': 1e-4,
    'ENC_MULT': 0.1,
    'WEIGHT_DECAY': 2e-4,
    'FOCAL_GAMMA': 1.5,
    'NUM_WORKERS': max(2, min(4, (os.cpu_count() or 2) - 1)),
    'EARLY_FUSION': False,
    'UNFREEZE_POLICY': 'layer4',
    'USE_TEMPERED_SAMPLER': True,
    'TEMPER_EXP': 0.5,
    'USE_TTA_VAL': False,
    'USE_TTA_TEST': True,
    'MAX_TRAIN_STEPS': None,
    'MAX_VAL_STEPS': None,
}

siam_run_dir = CLS_RUNS_ROOT / f"siam_resnet50_sz{BASE_CFG['INPUT_SIZE']}_seed{BASE_CFG['SEED']}"
out = train_one_run(siam_run_dir, BASE_CFG, device)
print('Val macro-F1:', out['best_val_f1'], '| Test macro-F1:', out['test_f1'])

---
## PART D — Fine-tune classifier on Ukraine (Phase 2)

In [ ]:
from types import SimpleNamespace
from train import finetune_on_kolega
from dataset import build_kolega_cls_manifest

PRETRAIN_CKPT = siam_run_dir / 'artifacts' / 'best.pth'
FT_DIR        = PRETRAIN_CKPT.parent / 'finetune_kolega'

ft_cfg = SimpleNamespace(
    seed=42, backbone='resnet50', num_classes=4, input_size=512,
    batch_train=48, batch_val=32, batch_test=32,
    epochs=14, warmup_epochs=4, patience=6,
    lr_head=7.5e-5, enc_mult=0.03, weight_decay=3e-4, focal_gamma=1.3,
    val_fold=1, test_fold=0,
    use_paired_jitter=False, jitter_bcs=(0.1,0.1,0.1), jitter_h=0.05,
    crop_scale=(0.98, 1.0),
    use_tta_val=False, use_tta_test=True,
)

model_ft, ft_ckpt, ft_log = finetune_on_kolega(
    pretrain_ckpt    = PRETRAIN_CKPT,
    kolega_root      = CLS_ROOT,
    ft_dir           = FT_DIR,
    device           = device,
    cfg              = ft_cfg,
    build_manifest_fn= build_kolega_cls_manifest,
)

### D1 — Post-training plots (classification)

In [ ]:
from eval import plot_learning_curves
plot_learning_curves(ft_log, run='last')

---
## PART E — Full Inference Pipeline

In [ ]:
from inference import run_inference

run_inference(
    pre_dir          = WAR_PRE_DIR,
    post_dir         = WAR_POST_DIR,
    seg_ckpt         = SEG_CKPT,
    cls_ckpt         = CLS_CKPT,
    device           = device,
    seg_input        = 1380,
    seg_scales       = (1.0, 1.2),
    seg_thr          = 0.35,
    min_component    = 64,
    split_touching_flag = True,
    use_densecrf     = True,
    cls_backbone     = 'resnet50',
    cls_input        = 256,
    use_tta_cls      = True,
    cls_use_mask     = True,
    use_ecc_align    = True,
)